# Monte Carlo Simulation & Statistical Modeling
**Author:** Abdellah Kahlaoui — Master of Applied Mathematics, FST Settat  
**GitHub:** [AbdellahKah/monte-carlo-statistical-modeling](https://github.com/AbdellahKah/monte-carlo-statistical-modeling)

---

This notebook walks through the full Monte Carlo pipeline:

1. Simulating probability distributions with 1,000,000 paths
2. Validating empirical moments against theoretical values
3. Proving CLT convergence rates experimentally
4. Pricing a European call option via Monte Carlo vs Black-Scholes
5. Interactive visualizations of all results

## 0. Setup

In [ ]:
import numpy as np
import plotly.io as pio

from monte_carlo import MonteCarloEngine
from analysis   import MCAnalyser
from plots      import (
    plot_distributions,
    plot_convergence,
    plot_option_convergence,
)

pio.renderers.default = "notebook"

# Reproducible engine — 1 million paths
ENGINE   = MonteCarloEngine(n_paths=1_000_000, seed=42)
ANALYSER = MCAnalyser(ENGINE)

print(f"Engine ready — {ENGINE.n_paths:,} paths | seed=42")

---
## 1. Simulating Probability Distributions

We simulate four distributions using **vectorized NumPy** — all 1,000,000 paths generated in a single array operation, making it orders of magnitude faster than a Python loop.

| Distribution | Parameters | Common use in quant finance |
|---|---|---|
| Normal | μ=0, σ=1 | Returns, noise, Z-scores |
| Log-Normal | μ=0, σ=0.5 | Asset prices (GBM) |
| Exponential | λ=2 | Inter-arrival times, default intensity |
| Poisson | λ=5 | Event counts, jump processes |

In [ ]:
# Run all four simulations
normal      = ENGINE.simulate_normal(mu=0.0, sigma=1.0)
lognormal   = ENGINE.simulate_lognormal(mu=0.0, sigma=0.5)
exponential = ENGINE.simulate_exponential(lam=2.0)
poisson     = ENGINE.simulate_poisson(lam=5.0)

# Print summary statistics
for sim in [normal, lognormal, exponential, poisson]:
    print(f"{sim.distribution:<14} "
          f"mean={sim.mean:>9.5f}  "
          f"std={sim.std:>9.5f}  "
          f"skew={sim.skewness:>7.4f}  "
          f"kurt={sim.kurtosis:>7.4f}")

### 1.1 Distribution Histograms vs Theoretical PDF

In [ ]:
fig = plot_distributions(ENGINE)
fig.show()

---
## 2. Statistical Validation

We compare empirical moments (mean, std, skewness) to their **closed-form theoretical values** and run formal goodness-of-fit tests:
- **KS test** (Kolmogorov-Smirnov) for continuous distributions
- **χ² test** (chi-squared) for the discrete Poisson distribution

All tests should **pass** (p-value > 0.05), confirming our sampler is correctly implemented.

In [ ]:
validation_results = ANALYSER.full_report()

### 2.1 Confidence Interval Coverage

The 95% CI on the mean (derived from the CLT: $\bar{x} \pm 1.96 \cdot \sigma/\sqrt{n}$) should contain the true mean for all distributions.

In [ ]:
print(f"{'Distribution':<16} {'95% CI':<32} {'True Mean':>10} {'Covers?':>8}")
print("-" * 68)

for r in validation_results:
    ci_str = f"({r.empirical_mean - 1.96*r.empirical_std/1000:.5f}, "\
             f"{r.empirical_mean + 1.96*r.empirical_std/1000:.5f})"
    print(f"{r.distribution:<16} {ci_str:<32} "
          f"{r.theoretical_mean:>10.5f} "
          f"{'Yes ✓' if r.ci_covers_true else 'No ✗':>8}")

---
## 3. Convergence Analysis — Proving the CLT

The **Central Limit Theorem** states that the standard error of the sample mean decays as:

$$SE(\bar{X}_n) = \frac{\sigma}{\sqrt{n}} \propto n^{-0.5}$$

In log-log space, this means the slope of $\log(SE)$ vs $\log(n)$ should be exactly **−0.50**.  
We verify this empirically across all four distributions.

In [ ]:
from analysis import convergence_summary
convergence_summary(ANALYSER)

### 3.1 Convergence Plot

In [ ]:
fig2 = plot_convergence(ANALYSER)
fig2.show()

---
## 4. European Call Option Pricing

We price a European call option under **Black-Scholes** assumptions using Monte Carlo:

$$S_T = S_0 \exp\left[(r - \tfrac{1}{2}\sigma^2)T + \sigma\sqrt{T}\, Z\right], \quad Z \sim \mathcal{N}(0,1)$$

$$C = e^{-rT} \, \mathbb{E}^\mathbb{Q}[\max(S_T - K,\, 0)]$$

The **analytical Black-Scholes price** (S₀=100, K=100, T=1yr, r=5%, σ=20%) is **10.4506**.  
We verify that our MC estimate converges to this value as n grows.

In [ ]:
result = ENGINE.price_european_call(S0=100, K=100, T=1, r=0.05, sigma=0.2)

bs_analytical = 10.4506  # Black-Scholes closed-form

print(f"MC Price       : {result['price']:.4f}")
print(f"Std Error      : {result['std_error']:.6f}")
print(f"95% CI         : ({result['ci_95'][0]:.4f}, {result['ci_95'][1]:.4f})")
print(f"BS Analytical  : {bs_analytical:.4f}")
print(f"Absolute Error : {abs(result['price'] - bs_analytical):.4f}")
print(f"Relative Error : {abs(result['price'] - bs_analytical)/bs_analytical*100:.3f}%")

### 4.1 Option Price Convergence Plot

In [ ]:
fig3 = plot_option_convergence(ENGINE)
fig3.show()

---
## 5. Key Takeaways

| Finding | Result |
|---|---|
| All 4 distributions pass goodness-of-fit tests | KS / χ² p-values >> 0.05 |
| CLT convergence rate | Slope ≈ −0.50 for all distributions |
| Option pricing accuracy (1M paths) | Error < 0.3% vs Black-Scholes |
| Vectorized NumPy performance | 1M paths simulated in < 1 second |

---

**Author:** Abdellah Kahlaoui  
**LinkedIn:** [linkedin.com/in/kahabdu1808](https://linkedin.com/in/kahabdu1808)  
**GitHub:** [github.com/AbdellahKah](https://github.com/AbdellahKah)